# 001 · try / except / finally

> **Syntax › Error Handling** · Python 3.14

Python uses exceptions for error handling just like C#, with almost the same structure: `try` / `except` / `finally`, plus an extra `else` block. The bigger difference is cultural — Python code uses exceptions *more* freely, following the "easier to ask forgiveness than permission" style.

**In this notebook you will learn:**

- `try` / `except` / `else` / `finally` and how they map to `try` / `catch` / `finally`
- the exception hierarchy (`BaseException` vs `Exception`) and the built-in exceptions that correspond to common .NET ones
- catching several exception types, and binding the exception with `as e`
- why a bare `except:` is dangerous
- EAFP vs LBYL — Python's preferred style
- printing tracebacks with the `traceback` module
- a first look at exception groups and `except*`

## C# mental model

| C# | Python |
|---|---|
| `try { } catch (FormatException ex) { }` | `try:` ... `except ValueError as ex:` |
| `catch (Exception ex) when (ex is A or B)` | `except (A, B) as ex:` |
| `catch { }` / `catch (Exception)` | `except Exception:` (avoid bare `except:`) |
| `finally { }` | `finally:` |
| — | `else:` (runs only if no exception was raised) |
| `catch (X ex) when (cond)` | no filter syntax — test inside and re-`raise` |
| `ex.Message` | `str(ex)` / `ex.args` |
| `ex.StackTrace` | `traceback.format_exc()` |
| `System.Exception` | `Exception` (and above it, `BaseException`) |

## Basic `try` / `except`

```csharp
// C#
try
{
    var n = int.Parse("forty-two");
}
catch (FormatException ex)
{
    Console.WriteLine($"bad input: {ex.Message}");
}
```

In Python, `catch` is spelled `except`, and the exception variable is bound with `as`:

In [ ]:
try:
    n = int("forty-two")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

print("program continues")

Exceptions you don't catch propagate up the call stack, exactly as in C#. `except` clauses are checked **top to bottom** and the first match wins, so put specific types before general ones:

In [ ]:
def parse_ratio(text: str) -> float:
    a, b = text.split("/")
    return int(a) / int(b)

for text in ("3/4", "1/0", "x/2", "nope"):
    try:
        print(text, "->", parse_ratio(text))
    except ZeroDivisionError:
        print(text, "-> division by zero")
    except ValueError as e:           # int() failed OR the unpacking a, b failed
        print(text, "->", f"{type(e).__name__}: {e}")

## `else` and `finally`

The full statement has four parts:

- `try` — the code that might fail
- `except` — runs if a matching exception was raised
- `else` — runs if **no** exception was raised (no C# equivalent)
- `finally` — **always** runs, exception or not (same as C#)

Why `else`? It keeps the `try` block minimal, so you don't accidentally catch exceptions from code you didn't intend to protect.

In [ ]:
def safe_divide(a: float, b: float) -> None:
    try:
        result = a / b
    except ZeroDivisionError:
        print("  except: cannot divide by zero")
    else:
        print(f"  else:   result is {result}")   # only on success
    finally:
        print("  finally: always runs")

print("10 / 2"); safe_divide(10, 2)
print("10 / 0"); safe_divide(10, 0)

`finally` also runs when the `try` block leaves via `return`, `break` or `continue` — same as C#:

In [ ]:
def load() -> str:
    try:
        return "data"
    finally:
        print("finally runs before the value is returned")

print(load())

⚠️ **Gotcha:** C# forbids `return` inside `finally` (compile error CS0157). Python allows it, but it silently **swallows** any in-flight exception. Since 3.14 the compiler emits a `SyntaxWarning` for `return`/`break`/`continue` in `finally` (PEP 765). Don't do it.

In [ ]:
import warnings

code = '''
def swallow():
    try:
        raise ValueError("lost!")
    finally:
        return "finally won"
'''
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    namespace = {}
    exec(compile(code, "<demo>", "exec"), namespace)

print(caught[0].category.__name__, "-", caught[0].message)
print(namespace["swallow"]())         # the ValueError disappeared

## The exception hierarchy

All exceptions derive from `BaseException`. Almost everything you'll catch derives from its subclass `Exception`. The few classes that sit **directly** under `BaseException` are signals that are usually *not* errors:

```
BaseException
 ├── SystemExit          ← raised by sys.exit()
 ├── KeyboardInterrupt   ← Ctrl+C / Jupyter "interrupt kernel"
 ├── GeneratorExit       ← used internally when closing generators
 ├── BaseExceptionGroup
 └── Exception           ← catch this (or something more specific)
      ├── ArithmeticError → ZeroDivisionError, OverflowError, ...
      ├── LookupError     → KeyError, IndexError
      ├── ValueError, TypeError, AttributeError, NameError, ...
      ├── OSError         → FileNotFoundError, PermissionError, TimeoutError, ...
      └── RuntimeError    → NotImplementedError, RecursionError
```

You can explore it with the method resolution order:

In [ ]:
print(KeyError.__mro__)
print(FileNotFoundError.__mro__)
print([cls.__name__ for cls in BaseException.__subclasses__()])   # in Jupyter, asyncio's CancelledError shows up too

## Built-in exceptions mapped to .NET

There is no one-to-one mapping, but these are the closest equivalents:

| Python | Typical cause | Closest .NET exception |
|---|---|---|
| `ValueError` | right type, bad value: `int("abc")` | `ArgumentException`, `FormatException` |
| `TypeError` | wrong type: `"a" + 1`, wrong number of args | `InvalidCastException`-ish, `ArgumentException` |
| `KeyError` | missing dict key | `KeyNotFoundException` |
| `IndexError` | list index out of range | `ArgumentOutOfRangeException`, `IndexOutOfRangeException` |
| `AttributeError` | missing attribute — including `None.foo` | `MissingMemberException`, `NullReferenceException` |
| `ZeroDivisionError` | `1 / 0` | `DivideByZeroException` |
| `FileNotFoundError` | open a missing file | `FileNotFoundException` |
| `PermissionError` | no access | `UnauthorizedAccessException` |
| `OSError` | general I/O failure | `IOException` |
| `NotImplementedError` | abstract / unfinished method | `NotImplementedException` |
| `RuntimeError` | generic "invalid state" | `InvalidOperationException` |
| `TimeoutError` | operation timed out | `TimeoutException` |
| `RecursionError` | recursion too deep | `StackOverflowException` (which C# can't catch!) |
| `StopIteration` | iterator exhausted | `MoveNext()` returning `false` |

In [ ]:
attempts = [
    lambda: int("abc"),
    lambda: "total: " + 5,
    lambda: {"a": 1}["b"],
    lambda: [1, 2, 3][10],
    lambda: None.upper(),
    lambda: 1 / 0,
    lambda: open("/definitely/not/here.txt"),
]

for attempt in attempts:
    try:
        attempt()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

## Catching multiple exception types

Pass a **tuple** of types to handle several in the same way. (C# has no direct syntax — you'd use an exception filter: `catch (Exception ex) when (ex is FormatException or OverflowException)`.)

In [ ]:
def to_int(value: object) -> int | None:
    try:
        return int(value)
    except (ValueError, TypeError) as e:
        print(f"  could not convert {value!r}: {type(e).__name__}")
        return None

print(to_int("42"))
print(to_int("forty-two"))
print(to_int(None))

Because exception classes form a hierarchy, catching a base class catches all subclasses — `LookupError` catches both `KeyError` and `IndexError`:

In [ ]:
for container, key in (({"a": 1}, "z"), ([1, 2], 5)):
    try:
        container[key]
    except LookupError as e:
        print(f"{type(e).__name__} is a LookupError: {e}")

**New in 3.14 (PEP 758):** the parentheses are optional when you **don't** use `as`: `except ValueError, TypeError:`. With `as`, parentheses are still required. Prefer the parenthesised form — it works in every Python 3 version and reads unambiguously.

In [ ]:
try:
    int(None)
except ValueError, TypeError:         # 3.14+ only
    print("caught without parentheses")

## The `as e` variable and what's in it

`str(e)` is the message (C#'s `ex.Message`); `e.args` is the tuple of constructor arguments.

⚠️ **Gotcha:** unlike C#, the `as` variable is **deleted** at the end of the `except` block (to avoid a reference cycle through the traceback). Copy it to another name if you need it later.

In [ ]:
try:
    {"a": 1}["missing"]
except KeyError as e:
    print("str(e): ", str(e))         # KeyError's str() includes quotes
    print("e.args: ", e.args)
    saved = e

try:
    print(e)
except NameError as err:
    print(f"{type(err).__name__}: {err}")
print("saved:", repr(saved))

## Exception filters: no `when` clause

C# lets you filter with `catch (HttpRequestException ex) when (ex.StatusCode == 404)`. Python has no filter syntax — check inside the handler and use a bare `raise` to re-raise the exception unchanged if it isn't yours *(see Error Handling › 002 · Raising and Custom Exceptions)*:

In [ ]:
import errno

def read_config(path: str) -> str:
    try:
        with open(path) as f:
            return f.read()
    except OSError as e:
        if e.errno == errno.ENOENT:   # "when (file not found)"
            return "<defaults>"
        raise                         # anything else propagates

print(read_config("/no/such/config.toml"))

## ⚠️ Bare `except:` pitfalls

A bare `except:` (or `except BaseException:`) catches **everything**, including `KeyboardInterrupt` and `SystemExit`. That means Ctrl+C or `sys.exit()` can be silently swallowed. Catch `Exception` (or, better, something specific) instead.

In [ ]:
import sys

def risky_bare():
    try:
        sys.exit(1)                   # a request to stop the program
    except:                           # noqa: E722 — catches SystemExit too!
        print("bare except swallowed SystemExit — the program keeps running")

def risky_exception():
    try:
        sys.exit(1)
    except Exception:                 # SystemExit is NOT an Exception
        print("never printed")

risky_bare()
try:
    risky_exception()
except SystemExit as e:
    print(f"SystemExit propagated correctly (code {e.code})")

Even `except Exception:` is too broad most of the time: it hides bugs like typos (`NameError`) and wrong types. Catch the narrowest type you can **handle meaningfully**, and log or re-raise the rest.

In [ ]:
def total_price(items: list[dict]) -> float:
    try:
        return sum(item["price"] for item in itemz)   # typo!
    except Exception:
        return 0.0                    # the NameError is hidden

print(total_price([{"price": 2.0}]))  # 0.0 — the bug is invisible

## EAFP vs LBYL

Two styles of defensive code:

- **LBYL** — *Look Before You Leap*: check first (`if key in d: ...`). Typical in C# (`TryGetValue`, `File.Exists`, `int.TryParse`).
- **EAFP** — *Easier to Ask Forgiveness than Permission*: just do it and handle the exception. Idiomatic Python.

In Python, `try` blocks are cheap when no exception is raised, and EAFP avoids race conditions (e.g. a file deleted between the `exists()` check and the `open()`).

```csharp
// C# — LBYL with Try* methods
if (int.TryParse(text, out var n)) { ... }
if (dict.TryGetValue(key, out var value)) { ... }
```

In [ ]:
stock = {"apple": 3, "bread": 0}

# LBYL
if "milk" in stock:
    print(stock["milk"])
else:
    print("LBYL: no milk")

# EAFP
try:
    print(stock["milk"])
except KeyError:
    print("EAFP: no milk")

# For this particular case, dict.get is cleanest
print("get:", stock.get("milk", 0))

# Python has no int.TryParse — the EAFP equivalent is a small helper
def try_parse_int(text: str) -> int | None:
    try:
        return int(text)
    except ValueError:
        return None

print(try_parse_int("123"), try_parse_int("12a"))

## Printing tracebacks: the `traceback` module

An uncaught exception prints a *traceback* (C#: stack trace), most recent call **last**. When you catch an exception but still want to log the trace, use the `traceback` module:

- `traceback.print_exc()` — print the current exception's traceback (to stderr)
- `traceback.format_exc()` — the same, as a string (C#: `ex.ToString()`)
- `traceback.format_exception(e)` — format a specific exception object

In [ ]:
import traceback

def level_two():
    return {"a": 1}["b"]

def level_one():
    return level_two()

try:
    level_one()
except KeyError:
    trace = traceback.format_exc()
    print(trace)

In real applications use the `logging` module: `logging.exception("message")` inside an `except` block logs the message **and** the traceback.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", stream=sys.stdout, force=True)

try:
    level_one()
except KeyError:
    logging.exception("lookup failed")

## Exception groups and `except*` (a first look)

Since 3.11, an `ExceptionGroup` bundles **several** exceptions raised at once — the equivalent of .NET's `AggregateException`. It appears mainly in concurrent code, e.g. `asyncio.TaskGroup` *(see Async › 002 · Running Tasks Concurrently)*.

`except*` handles **each matching part** of a group; unmatched parts continue to propagate. Unlike `except`, **several** `except*` clauses can run for one group, and you can't mix `except` and `except*` in the same `try`.

```csharp
// C#
catch (AggregateException ae)
{
    foreach (var inner in ae.InnerExceptions) { ... }
}
```

In [ ]:
def run_all():
    raise ExceptionGroup("several jobs failed", [
        ValueError("job 1: bad input"),
        TypeError("job 2: wrong type"),
        ValueError("job 3: bad input"),
    ])

try:
    run_all()
except* ValueError as group:          # group is an ExceptionGroup of just the ValueErrors
    for e in group.exceptions:
        print("value problem:", e)
except* TypeError as group:
    for e in group.exceptions:
        print("type problem: ", e)

## Summary

| Task | C# | Python |
|---|---|---|
| Catch | `catch (FormatException ex)` | `except ValueError as e:` |
| Catch several | `catch (Exception ex) when (ex is A or B)` | `except (A, B) as e:` (3.14: `except A, B:` without `as`) |
| Catch all errors | `catch (Exception)` | `except Exception:` — never a bare `except:` |
| Success-only block | — | `else:` |
| Always run | `finally` | `finally:` |
| Filter | `when (cond)` | `if not cond: raise` inside the handler |
| Message | `ex.Message` | `str(e)` |
| Stack trace | `ex.StackTrace` / `ex.ToString()` | `traceback.format_exc()` |
| Try-parse | `int.TryParse` | EAFP: `try: int(s) except ValueError:` |
| Aggregate | `AggregateException` | `ExceptionGroup` + `except*` |

## Exercises

1. Write `safe_get(items, index)` that returns `items[index]`, or `None` if the index is out of range or `items` is not indexable.
2. Write `read_number(text)` using `try`/`except`/`else`/`finally` that prints `"parsed N"` on success, `"invalid"` on failure, and `"done"` in every case.
3. Rewrite this LBYL code in EAFP style:
   ```python
   if "port" in config and config["port"].isdigit():
       port = int(config["port"])
   else:
       port = 8080
   ```
4. Why should you avoid `except:` without a type? Name two exceptions it would catch that `except Exception:` would not.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def safe_get(items, index):
    try:
        return items[index]
    except (IndexError, TypeError):
        return None

# 2
def read_number(text: str) -> None:
    try:
        n = int(text)
    except ValueError:
        print("invalid")
    else:
        print(f"parsed {n}")
    finally:
        print("done")

# 3
try:
    port = int(config["port"])
except (KeyError, ValueError):
    port = 8080

# 4 — it also catches BaseException subclasses that aren't errors,
#     e.g. KeyboardInterrupt (Ctrl+C) and SystemExit (sys.exit()),
#     so the user can't stop the program and exit requests are ignored.
```

</details>